In [ ]:
import glob
import os
import shutil


In [ ]:
import os
print(os.getcwd())
%cd /home/zmirikha/GitHub/rnd_q2
print(os.getcwd())

In [ ]:
#Remove positive QCed samples from negative samples in v1 dataset and add them to a validation dataset
v1_negative_dir = "data/fhr4/final_data_21_classifier_2_channels_std/neg"
v1_1_validation_dir = "data/fhr4/final_data_v1.2/validation/pos"
v1_1_neg_dir = "data/fhr4/final_data_v1.2/training/neg"
remove_samples = [
    115892170,
    116673420,
    119393230,
    121230787,
    125938230,
    133662610,
    143069720,
    165940680,
    168731220,
    140987670,
    139915710,
    131142140,
    130736360,
    130167860,
    129523310,
    128453820,
    127133680,
    124551540,
    119253330,
    119142370]

for  normalization_type in ["std_normalized", "patch_normalized"]:
    for f in glob.glob(os.path.join(v1_negative_dir,normalization_type, "*.npy")):
        base = os.path.basename(f)
        sample_number = int(base.split(".")[0][2:])
        if sample_number in remove_samples:
            print(f"Moving {base} to validation set")
            dst = os.path.join(v1_1_validation_dir,normalization_type, "fhr4_" + base.split(".")[0] + ".npy")
            os.makedirs(os.path.dirname(dst), exist_ok=True)
            shutil.copy(os.path.join(v1_negative_dir,normalization_type, base), dst)
        else:
            dst = os.path.join(v1_1_neg_dir,normalization_type, "fhr4_" + base.split(".")[0] + ".npy")
            os.makedirs(os.path.dirname(dst), exist_ok=True)
            shutil.copy(os.path.join(v1_negative_dir,normalization_type, base), dst)


In [ ]:
#16 samples from above
#add 7 samples from fhr3 to validation data

#add some Qced samples from fhr6 to negative training data

In [ ]:
v1_1_neg_dir = "data/fhr4/final_data_v1.2/training/neg"
v1_FP_dir = "./data/fhr6/"
FP_samples = [16150,
22899,
960649,
1024650,
896900,
798150,
650900,
652900,
305900,
175149,
10651,
11400,
10150,
4400,
891900,
826650,
758650,
684651,
654651,
562650,
541400,
16150,
21150,
106400,
145400]
for fld_name in ["pos_patch_normalized_qced", "pos_std_normalized_qced"]:
    for f in glob.glob(os.path.join(v1_FP_dir,fld_name, "*/*.npy")):
        base = os.path.basename(f)
        sample_number = int(base.split(".")[0][1:])
        if sample_number in FP_samples:
            print(f"Copying {base} to negative training set")
            norm_type_fld = "patch_normalized" if "patch" in fld_name else "std_normalized"
            dst = os.path.join(v1_1_neg_dir,norm_type_fld, "fhr6_" + base.split(".")[0] + ".npy")
            os.makedirs(os.path.dirname(dst), exist_ok=True)
            shutil.copy(f, dst)


In [ ]:
import os
import glob
import json
import numpy as np
import matplotlib.pyplot as plt

root = "./data/fhr4/"
for normalization_type_fld in ["pos_std_normalized", "pos_patch_normalized"]:
    original_pos_dir = os.path.join(root, normalization_type_fld+"_all")
    augmented_pos_dir = os.path.join(root, "final_data_v1.2/training/pos/", normalization_type_fld)
    metadata_path = os.path.join(root, "final_data_binary_classifier/dent_tracks_indices_cleaned_ae.json")
    metadata_dict = json.load(open(metadata_path, "r"))

    pos_fns = glob.glob(original_pos_dir + "/*.npy")[-15:]
    for i, fn in enumerate(pos_fns):
        print(f"Processing {fn} ({i+1}/{len(pos_fns)})")
        pos_sample = np.load(fn)
        filename = fn.split('/')[-1].split('.')[0]
        if filename not in metadata_dict.keys():
            print(f"Skipping {filename} as not in metadata")
            continue
        else:
            track_idx = metadata_dict[filename]
            os.makedirs(augmented_pos_dir, exist_ok=True)

            # Save original
            np.save(os.path.join(augmented_pos_dir, f"{filename}_track_{track_idx}.npy"), pos_sample)
            qc_output_dir = os.path.join(augmented_pos_dir, "qc")
            os.makedirs(qc_output_dir, exist_ok=True)
            plt.imshow(
                pos_sample.T, cmap="inferno", origin="lower", aspect="auto",
                interpolation="nearest",
            )
            plt.axis("off")
            plt.savefig(
                os.path.join(qc_output_dir, f"{filename}_track_{track_idx}.png"),
                bbox_inches="tight", pad_inches=0,
            )
            plt.close()
                
            # Vertical roll augmentations
            for i, shift_v in enumerate([2, 4, 6, 8, 10, 12, 14, 16, 18]):
                rolled_v = np.roll(pos_sample, shift_v, axis=0)  # Roll vertically

                # Horizontal roll augmentations
                for j, shift_h in enumerate([-20, 0, 20]):  # extend if you want more h-rolls
                    rolled_vh = np.roll(rolled_v, shift_h, axis=1)

                    # -------------------------------
                    # ✅ Gaussian noise augmentation
                    # -------------------------------
                    noise_sigma = 0.02  # adjust strength (0.01–0.05 is typical)
                    noisy = rolled_vh + np.random.normal(0, noise_sigma, rolled_vh.shape)
                    noisy = np.clip(noisy, 0, 1)  # keep in [0,1]

                    # -------------------------------
                    # ✅ Intensity scaling augmentation
                    # -------------------------------
                    scale_factor = np.random.uniform(0.9, 1.1)  # ±10% scaling
                    scaled = rolled_vh * scale_factor
                    scaled = np.clip(scaled, 0, 1)

                    # Save baseline rolled
                    track_idx_new = [(tridx+shift_v) % 20 for tridx in track_idx]
                    base_name = f"{filename}_track_{track_idx_new}_hroll_{shift_h}"

                    for aug_type, aug_array in [("orig", rolled_vh),
                                                ("noise", noisy),
                                                ("scaled", scaled)]:
                        aug_filename = f"{base_name}_{aug_type}.npy"
                        np.save(os.path.join(augmented_pos_dir, aug_filename), aug_array)
                        plt.imshow(
                            aug_array.T, cmap="inferno", origin="lower", aspect="auto",
                            interpolation="nearest",
                        )
                        plt.axis("off")
                        plt.savefig(
                            os.path.join(qc_output_dir, f"{aug_filename.split('.')[0]}.png"),
                            bbox_inches="tight", pad_inches=0,
                        )
                        plt.close()


In [5]:
import glob
import shutil
import os
file_lists = glob.glob("/home/zmirikha/GitHub/rnd_q2/data/fhr6/fhr4/final_data_v1.3/training/neg/std_normalized/qc/*.png")
src1 = "/home/zmirikha/GitHub/rnd_q2/data/fhr6/fhr4/final_data_v1.3/training/neg/std_normalized"
dst1 = "/home/zmirikha/GitHub/rnd_q2/data/fhr4/final_data_v1.3/training/neg/std_normalized"
for fn in file_lists: 
    src1_fn =   os.path.join(src1,"d"+fn.split("/")[-1].replace(".png", ".npy"))
    shutil.copy(src1_fn, dst1)
src2 = "/home/zmirikha/GitHub/rnd_q2/data/fhr6/fhr4/final_data_v1.3/training/neg/patch_normalized"
dst2 = "/home/zmirikha/GitHub/rnd_q2/data/fhr4/final_data_v1.3/training/neg/patch_normalized"
for fn in file_lists: 
    src2_fn =   os.path.join(src2,"d"+fn.split("/")[-1].replace(".png", ".npy"))
    shutil.copy(src2_fn, dst2)
